In [1]:
"""
Ячейка 1: Полный pipeline — имитация работы бота
"""
from sentence_transformers import SentenceTransformer, util
import torch
import time
import json

# ── 1. Загрузка моделей ──
print("📥 Загрузка моделей...")
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Для классификатора — используй обученную модель из Шага 4
# Здесь — упрощённая версия через embeddings + cosine similarity
print("✅ Модели загружены")

# ── 2. Подготовка данных ──
# FAQ
faq_data = [
    {"triggers": ["как оформить возврат", "хочу вернуть товар", "возврат товара"],
     "answer": "Для возврата товара: 1) Перейдите в 'Мои заказы'. 2) Выберите товар. 3) Нажмите 'Оформить возврат'. Возврат возможен в течение 14 дней.",
     "category": "return_exchange"},
    {"triggers": ["сколько стоит доставка", "стоимость доставки", "цена доставки"],
     "answer": "Стоимость доставки зависит от региона. Бесплатная доставка при заказе от 5000₽. Москва и СПб: от 300₽, регионы: от 400₽.",
     "category": "delivery"},
    {"triggers": ["режим работы", "во сколько работаете", "часы работы"],
     "answer": "Интернет-магазин работает круглосуточно. Служба поддержки: 9:00–21:00 МСК, без выходных.",
     "category": "general_info"},
]

# База знаний (чанки)
knowledge_base_chunks = [
    "Возврат товара надлежащего качества возможен в течение 14 дней с момента получения.",
    "Доставка осуществляется курьерской службой СДЭК и Почтой России.",
    "Бесплатная доставка при заказе от 5000 рублей. Срок: Москва 1-2 дня, регионы 3-7 дней.",
    "Оплата возможна картой Visa, MasterCard, МИР, через СБП, Apple Pay, Google Pay.",
    "Возврат денег осуществляется в течение 10 рабочих дней на карту оплаты.",
    "Гарантия на электронику — 12 месяцев. На одежду — 30 дней.",
    "Промокод вводится на странице оформления заказа в поле 'Промокод'.",
    "Связаться с нами: 8-800-123-45-67, support@shop.ru, чат на сайте.",
]

# Заказы
orders_db = {
    "123456": {"status": "Отправлен", "delivery": "10.04.2026", "track": "RU123456789CN"},
    "789012": {"status": "Доставлен", "delivery": "05.04.2026", "track": "RU789012345CN"},
    "555555": {"status": "Собирается", "delivery": "15.04.2026", "track": None},
}

# Индексация
faq_triggers_flat = []
faq_mapping = []
for faq in faq_data:
    for trigger in faq["triggers"]:
        faq_triggers_flat.append(trigger)
        faq_mapping.append(faq)

faq_embeddings = embedder.encode(faq_triggers_flat, convert_to_tensor=True)
kb_embeddings = embedder.encode(knowledge_base_chunks, convert_to_tensor=True)

print(f"✅ FAQ: {len(faq_triggers_flat)} триггеров")
print(f"✅ KB:  {len(knowledge_base_chunks)} чанков")


# ── 3. Pipeline функции ──

def check_meta_intent(text: str) -> dict | None:
    """Проверка мета-интентов (паттерны)"""
    text_lower = text.lower().strip()
    
    greetings = ["привет", "здравствуйте", "добрый день", "добрый вечер", "хай", "hello"]
    goodbyes = ["пока", "до свидания", "спасибо, всё", "всё, спасибо"]
    operator_requests = ["оператор", "живой человек", "позовите менеджера", "соедините с оператором"]
    
    for g in greetings:
        if g in text_lower:
            return {"intent": "greeting", "response": "Здравствуйте! 👋 Я бот поддержки. Чем могу помочь?\n\n• Статус заказа\n• Возврат товара\n• Доставка\n• Другой вопрос"}
    
    for g in goodbyes:
        if g in text_lower:
            return {"intent": "goodbye", "response": "Спасибо за обращение! Если будут вопросы — пишите. До свидания! 👋"}
    
    for o in operator_requests:
        if o in text_lower:
            return {"intent": "operator", "response": "🔄 Переключаю на оператора. Ожидайте, пожалуйста..."}
    
    return None

def check_faq(text: str, threshold: float = 0.85) -> dict | None:
    """Проверка FAQ по similarity"""
    query_emb = embedder.encode(text, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, faq_embeddings)[0]
    max_idx = scores.argmax().item()
    max_score = scores[max_idx].item()
    
    if max_score >= threshold:
        faq = faq_mapping[max_idx]
        return {"source": "faq", "answer": faq["answer"], "similarity": max_score, "category": faq["category"]}
    return None

def check_order(text: str) -> dict | None:
    """Извлечение номера заказа и получение статуса"""
    import re
    match = re.search(r'(\d{6})', text)
    if match:
        order_num = match.group(1)
        if order_num in orders_db:
            order = orders_db[order_num]
            response = f"📦 Заказ #{order_num}\n"
            response += f"Статус: {order['status']}\n"
            response += f"Ожидаемая доставка: {order['delivery']}\n"
            if order['track']:
                response += f"Трек-номер: {order['track']}"
            return {"source": "order_api", "answer": response, "order_number": order_num}
        else:
            return {"source": "order_api", "answer": f"❌ Заказ #{order_num} не найден. Проверьте номер.", "order_number": order_num}
    return None

def query_rag(text: str, threshold: float = 0.70, top_k: int = 3) -> dict:
    """RAG: поиск по базе знаний"""
    query_emb = embedder.encode(text, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, kb_embeddings)[0]
    
    top_indices = scores.argsort(descending=True)[:top_k]
    results = []
    for idx in top_indices:
        sim = scores[idx.item()].item()
        if sim >= threshold:
            results.append({"text": knowledge_base_chunks[idx.item()], "similarity": sim})
    
    if results:
        answer = results[0]["text"]
        return {"source": "rag", "answer": answer, "similarity": results[0]["similarity"], "chunks_found": len(results)}
    else:
        return {"source": "rag", "answer": None, "similarity": scores.max().item(), "chunks_found": 0}


# ── 4. Главный pipeline ──

def process_message(text: str, session: dict) -> dict:
    """
    Полный pipeline обработки сообщения
    Возвращает: {"answer": str, "source": str, "details": dict}
    """
    start = time.perf_counter()
    
    # Уровень 0: Мета-интенты
    meta = check_meta_intent(text)
    if meta:
        elapsed = (time.perf_counter() - start) * 1000
        return {"answer": meta["response"], "source": "meta", "time_ms": elapsed, "details": meta}
    
    # Уровень 1: FAQ
    faq_result = check_faq(text)
    if faq_result:
        elapsed = (time.perf_counter() - start) * 1000
        return {"answer": faq_result["answer"], "source": "faq", "time_ms": elapsed, "details": faq_result}
    
    # Уровень 1.5: Проверка заказа (если есть номер)
    order_result = check_order(text)
    if order_result:
        elapsed = (time.perf_counter() - start) * 1000
        return {"answer": order_result["answer"], "source": "order_api", "time_ms": elapsed, "details": order_result}
    
    # Уровень 2: RAG
    rag_result = query_rag(text)
    if rag_result["answer"]:
        elapsed = (time.perf_counter() - start) * 1000
        return {"answer": rag_result["answer"], "source": "rag", "time_ms": elapsed, "details": rag_result}
    
    # Уровень 3: Fallback
    session["fail_count"] = session.get("fail_count", 0) + 1
    elapsed = (time.perf_counter() - start) * 1000
    
    if session["fail_count"] >= 2:
        return {
            "answer": "Похоже, я не могу помочь с этим вопросом. Давайте подключу оператора? 🔄",
            "source": "escalation",
            "time_ms": elapsed,
            "details": {"fail_count": session["fail_count"]}
        }
    
    return {
        "answer": "Не совсем понял ваш вопрос. Попробуйте переформулировать или выберите тему:\n• Статус заказа\n• Возврат\n• Доставка\n• Оператор",
        "source": "fallback",
        "time_ms": elapsed,
        "details": {"fail_count": session["fail_count"]}
    }


# ── 5. Тестирование pipeline ──

test_messages = [
    "Привет!",
    "Как оформить возврат товара?",
    "Где мой заказ 123456?",
    "Где заказ 999999?",
    "Какая гарантия на телефон?",
    "Можно оплатить при получении?",
    "ывлаодыфва",
    "ещё раз непонятно",
    "Оператор",
    "Сколько стоит доставка?",
    "До свидания!",
]

session = {}

print("=" * 70)
print("🤖 ТЕСТИРОВАНИЕ PIPELINE ЧАТ-БОТА")
print("=" * 70)

for msg in test_messages:
    result = process_message(msg, session)
    
    print(f"\n👤 Клиент: {msg}")
    print(f"🤖 Бот [{result['source']}] ({result['time_ms']:.1f}ms):")
    print(f"   {result['answer']}")
    
    if result["source"] == "escalation":
        session["fail_count"] = 0  # сброс после эскалации

📥 Загрузка моделей...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Модели загружены
✅ FAQ: 9 триггеров
✅ KB:  8 чанков
🤖 ТЕСТИРОВАНИЕ PIPELINE ЧАТ-БОТА

👤 Клиент: Привет!
🤖 Бот [meta] (0.0ms):
   Здравствуйте! 👋 Я бот поддержки. Чем могу помочь?

• Статус заказа
• Возврат товара
• Доставка
• Другой вопрос

👤 Клиент: Как оформить возврат товара?
🤖 Бот [faq] (68.6ms):
   Для возврата товара: 1) Перейдите в 'Мои заказы'. 2) Выберите товар. 3) Нажмите 'Оформить возврат'. Возврат возможен в течение 14 дней.

👤 Клиент: Где мой заказ 123456?
🤖 Бот [order_api] (20.2ms):
   📦 Заказ #123456
Статус: Отправлен
Ожидаемая доставка: 10.04.2026
Трек-номер: RU123456789CN

👤 Клиент: Где заказ 999999?
🤖 Бот [order_api] (18.1ms):
   ❌ Заказ #999999 не найден. Проверьте номер.

👤 Клиент: Какая гарантия на телефон?
🤖 Бот [fallback] (75.7ms):
   Не совсем понял ваш вопрос. Попробуйте переформулировать или выберите тему:
• Статус заказа
• Возврат
• Доставка
• Оператор

👤 Клиент: Можно оплатить при получении?
🤖 Бот [escalation] (48.1ms):
   Похоже, я не могу помочь с этим в